In [ ]:
import pandas as pd
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import json


PROJECT_ROOT = Path.cwd().parent
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "sample_features.csv"

FORECAST_PATH = PROJECT_ROOT / "models" / "artifacts" / "xgb_forecast.csv"
METRICS_PATH = PROJECT_ROOT / "models" / "metrics" / "xgb_metrics.json"


df = pd.read_csv(FEATURES_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)

y = df["value"]
X = df.drop(columns=["value", "date"])  


df_clean = pd.concat([X, y], axis=1).dropna()
X = df_clean.drop(columns=["value"])
y = df_clean["value"]

print(f"[INFO] Shape après suppression des NaN : X={X.shape}, y={y.shape}")

model = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
model.fit(X, y)


y_pred = model.predict(X)

mae = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

metrics = {
    "MAE": mae,
    "RMSE": rmse,
    "num_samples": len(y)
}

print("Métriques XGBoost baseline :", metrics)

forecast = df_clean[["date", "value"]].copy()
forecast["yhat"] = y_pred


FORECAST_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)

forecast.to_csv(FORECAST_PATH, index=False)
with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Forecast sauvegardé : {FORECAST_PATH}")
print(f"Métriques sauvegardées : {METRICS_PATH}")

